In [6]:
# IterNormIndividualAuto.py
# 一个完全依赖 PyTorch Autograd 的版本

import torch
import torch.nn as nn
from torch.nn import Parameter

class IterNormIndividualAuto(nn.Module):
    def __init__(self, num_channels, seq_len, T=5, eps=1e-3, momentum=0.1, affine=True, *args, **kwargs):
        super(IterNormIndividualAuto, self).__init__()
        self.num_channels = num_channels
        self.seq_len = seq_len
        self.T = T
        # 增大eps以保证数值稳定性
        self.eps = eps
        self.momentum = momentum
        self.affine = affine
        
        if self.affine:
            self.weight = Parameter(torch.ones(1, self.num_channels, self.seq_len))
            self.bias = Parameter(torch.zeros(1, self.num_channels, self.seq_len))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)

        self.register_buffer('running_mean', torch.zeros(self.num_channels, self.seq_len, 1))
        initial_wm = torch.eye(self.seq_len).expand(self.num_channels, self.seq_len, self.seq_len).clone()
        self.register_buffer('running_wm', initial_wm)
        
        self.reset_parameters()

    def reset_parameters(self):
        if self.affine:
            nn.init.ones_(self.weight)
            nn.init.zeros_(self.bias)

    def forward(self, X: torch.Tensor):
        assert X.dim() == 3, f"Input must be a 3D tensor (B, C, L), but got {X.dim()}D"
        assert X.size(1) == self.num_channels
        assert X.size(2) == self.seq_len
        
        # 将原 autograd.Function.forward 的逻辑直接嵌入此处
        B, C, L = X.shape
        x_reshaped = X.permute(1, 2, 0).contiguous() # (C, L, B)
        
        if self.training:
            # 1. 计算批次统计量
            mean = x_reshaped.mean(-1, keepdim=True)  # (C, L, 1)
            xc = x_reshaped - mean
            
            I = torch.eye(L, device=X.device, dtype=X.dtype).expand(C, L, L)
            Sigma = torch.baddbmm(I.mul(self.eps), xc, xc.transpose(1, 2), beta=1., alpha=1./B)
            
            rTr = (Sigma * I).sum((1, 2), keepdim=True).reciprocal_()
            Sigma_N = Sigma * rTr
            
            # 迭代计算白化矩阵
            P = I.clone()
            for _ in range(self.T):
                P_cubed = torch.matrix_power(P, 3)
                P = torch.baddbmm(P.mul(1.5), P_cubed, Sigma_N, beta=1., alpha=-0.5)
            
            wm = P.mul(rTr.sqrt()) # (C, L, L)
            
            # 2. 更新全局统计量
            # 使用 .detach() 来更新 running_mean 和 running_wm，
            # 因为这些统计量的更新过程不应计入反向传播的计算图中。
            with torch.no_grad():
                self.running_mean.copy_(self.momentum * mean + (1. - self.momentum) * self.running_mean)
                self.running_wm.copy_(self.momentum * wm + (1. - self.momentum) * self.running_wm)
        else:
            # 推理模式，直接使用全局统计量
            mean = self.running_mean
            wm = self.running_wm
            xc = x_reshaped - mean
            
        # 3. 应用白化变换
        xn = wm.matmul(xc) # (C, L, L) @ (C, L, B) -> (C, L, B)
        Xn = xn.permute(2, 0, 1).contiguous() # (B, C, L)
        
        # 4. 应用仿射变换
        if self.affine:
            return Xn * self.weight + self.bias
        else:
            return Xn

    def extra_repr(self):
        return 'num_channels={num_channels}, seq_len={seq_len}, T={T}, eps={eps}, momentum={momentum}, affine={affine}'.format(**self.__dict__)

In [7]:
def run_experiment_1():
    print("\n" + "="*25 + " 实验一: 白化有效性数值验证 " + "="*25)
    
    def create_correlated_data(B, C, L):
        X = torch.randn(B, C, L)
        X[:, 0, :] = 0
        X[:, 1, :] = 0
        for t in range(1, L):
            X[:, 0, t] = 0.9 * X[:, 0, t-1] + 0.4 * torch.randn(B)
            X[:, 1, t] = -0.9 * X[:, 1, t-1] + 0.4 * torch.randn(B)
        return X

    def analyze_covariance(data, title):
        C = data.shape[1]
        print(title)
        for i in range(C):
            channel_data = data[:, i, :].contiguous()
            B, L = channel_data.shape
            mean = channel_data.mean(dim=0, keepdim=True)
            centered_data = channel_data - mean
            covariance = (1.0 / (B - 1)) * centered_data.t() @ centered_data
            
            diag_elements = torch.diag(covariance)
            num_elements = covariance.numel()
            num_diag = diag_elements.numel()
            mean_off_diag = (torch.sum(torch.abs(covariance)) - torch.sum(torch.abs(diag_elements))) / (num_elements - num_diag)
            mean_diag = torch.mean(diag_elements)
            
            print(f"  --- Channel {i} ---")
            print(f"  - 对角线元素均值 (应接近 1.0): {mean_diag:.6f}")
            print(f"  - 非对角线元素绝对值均值 (应接近 0.0): {mean_off_diag:.6f}")
        print("-" * 70)

    B, C, L = 128, 3, 16 # 0: pos-corr, 1: neg-corr, 2: white-noise
    input_data = create_correlated_data(B, C, L)
    
    model = IterNormIndividualAuto(num_channels=C, seq_len=L, eps=1e-3, T=8)
    model.train()
    
    output_data = model(input_data)
    
    analyze_covariance(input_data, "输入数据协方差:")
    analyze_covariance(output_data, "输出数据协方差 (经 IterNormIndividualAuto):")


In [8]:
run_experiment_1()


========================= 实验一: 白化有效性数值验证 =========================
输入数据协方差:
  --- Channel 0 ---
  - 对角线元素均值 (应接近 1.0): 0.600652
  - 非对角线元素绝对值均值 (应接近 0.0): 0.311050
  --- Channel 1 ---
  - 对角线元素均值 (应接近 1.0): 0.601009
  - 非对角线元素绝对值均值 (应接近 0.0): 0.296390
  --- Channel 2 ---
  - 对角线元素均值 (应接近 1.0): 0.952506
  - 非对角线元素绝对值均值 (应接近 0.0): 0.066062
----------------------------------------------------------------------
输出数据协方差 (经 IterNormIndividualAuto):
  --- Channel 0 ---
  - 对角线元素均值 (应接近 1.0): 0.917834
  - 非对角线元素绝对值均值 (应接近 0.0): 0.016549
  --- Channel 1 ---
  - 对角线元素均值 (应接近 1.0): 0.904944
  - 非对角线元素绝对值均值 (应接近 0.0): 0.016602
  --- Channel 2 ---
  - 对角线元素均值 (应接近 1.0): 1.006660
  - 非对角线元素绝对值均值 (应接近 0.0): 0.000090
----------------------------------------------------------------------


In [9]:
from torch.autograd import gradcheck

def run_experiment_2():
    print("\n" + "="*25 + " 实验二: autograd 梯度校验 " + "="*25)
    
    # gradcheck 对数值精度要求高, 使用 double 类型
    # 使用小批量、短序列、良态输入(白噪声)以保证 gradcheck 稳定运行
    B, C, L = 2, 2, 4
    # 使用随机数据而非相关数据，避免数值不稳定影响梯度检查本身
    input_data = torch.randn(B, C, L, dtype=torch.double, requires_grad=True)

    # 模块也必须是 double 类型
    model = IterNormIndividualAuto(num_channels=C, seq_len=L, eps=1e-3).double()
    model.train()

    # gradcheck 会将 model 视为一个函数
    is_correct = gradcheck(model, input_data, eps=1e-6, atol=1e-4)
    
    print(f"梯度检查是否通过: {is_correct}")
    if is_correct:
        print("结论: autograd 成功为此模块计算了正确的梯度。底层逻辑正确。")
    else:
        print("结论: 梯度检查失败！模块的 forward 实现存在 autograd 无法处理的问题。")
        


In [10]:
run_experiment_2()


========================= 实验二: autograd 梯度校验 =========================
梯度检查是否通过: True
结论: autograd 成功为此模块计算了正确的梯度。底层逻辑正确。


In [11]:
def run_experiment_3():
    print("\n" + "="*25 + " 实验三: 推理(eval)模式行为验证 " + "="*25)
    B, C, L = 8, 3, 16
    
    model = IterNormIndividualAuto(num_channels=C, seq_len=L, eps=1e-3)
    
    # 1. 模拟训练过程，填充 running statistics
    print("--- 模拟训练以填充运行统计量...")
    model.train()
    for _ in range(5):
        # 每步都用新数据
        dummy_input = torch.randn(B, C, L)
        _ = model(dummy_input)

    # 2. 切换到评估模式
    model.eval()
    print("--- 已切换到 eval() 模式 ---")
    
    # 3. 确定性验证
    print("\n--- 3a. 确定性验证 ---")
    test_input = torch.randn(B, C, L)
    output1 = model(test_input)
    output2 = model(test_input)
    is_deterministic = torch.allclose(output1, output2)
    print(f"两次相同输入的输出是否一致: {is_deterministic}")
    if is_deterministic:
        print("结论: 模块在 eval 模式下行为是确定性的。")
    else:
        print("结论: 模块在 eval 模式下行为不确定！存在严重错误。")

    # 4. 正确性验证
    print("\n--- 3b. 手动计算验证 ---")
    with torch.no_grad():
        # 模型输出
        model_output = model(test_input)
        
        # 手动计算
        running_wm = model.running_wm
        running_mean = model.running_mean
        
        x_reshaped = test_input.permute(1, 2, 0).contiguous() # (C, L, B)
        xc = x_reshaped - running_mean
        xn = running_wm.matmul(xc)
        manual_output = xn.permute(2, 0, 1).contiguous()
        
        # 如果模型有仿射层，也需手动应用
        if model.affine:
            manual_output = manual_output * model.weight + model.bias

        is_correct = torch.allclose(model_output, manual_output, atol=1e-6)
        print(f"手动计算结果是否与模型输出一致: {is_correct}")
        if is_correct:
            print("结论: 模块在 eval 模式下正确地使用了其运行统计量。")
        else:
            print("结论: 模块在 eval 模式下的计算逻辑不正确！")



In [12]:
run_experiment_3()


========================= 实验三: 推理(eval)模式行为验证 =========================
--- 模拟训练以填充运行统计量...
--- 已切换到 eval() 模式 ---

--- 3a. 确定性验证 ---
两次相同输入的输出是否一致: True
结论: 模块在 eval 模式下行为是确定性的。

--- 3b. 手动计算验证 ---
手动计算结果是否与模型输出一致: True
结论: 模块在 eval 模式下正确地使用了其运行统计量。


In [16]:
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

def run_experiment_4():
    print("\n" + "="*25 + " 实验四: 下游任务收敛性对比实验 " + "="*25)

    # 1. 创建数据集
    def create_classification_data(num_samples, C, L):
        X = torch.zeros(num_samples, C, L)
        # 类别0: 正相关, 类别1: 负相关, 类别2: 白噪声
        Y = torch.randint(0, 3, (num_samples,))
        for i in range(num_samples):
            if Y[i] == 0: # 正相关
                for t in range(1, L): X[i, :, t] = 0.9 * X[i, :, t-1] + 0.4 * torch.randn(C)
            elif Y[i] == 1: # 负相关
                for t in range(1, L): X[i, :, t] = -0.9 * X[i, :, t-1] + 0.4 * torch.randn(C)
            else: # 白噪声
                X[i] = torch.randn(C, L)
        return X, Y
    
    # 2. 定义模型
    class SimpleCNN(nn.Module):
        def __init__(self, C, L, use_iternorm=False):
            super().__init__()
            self.use_iternorm = use_iternorm
            self.conv1 = nn.Conv1d(C, 16, kernel_size=3, padding=1)
            if use_iternorm:
                self.norm1 = IterNormIndividualAuto(num_channels=16, seq_len=L)
            self.relu = nn.ReLU()
            self.pool = nn.AdaptiveAvgPool1d(1)
            self.fc = nn.Linear(16, 3) # 3个类别
        def forward(self, x):
            x = self.conv1(x)
            if self.use_iternorm:
                x = self.norm1(x)
            x = self.relu(x)
            x = self.pool(x).squeeze(-1)
            x = self.fc(x)
            return x

    # 3. 训练过程
    def train_model(model, loader, epochs=20):
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        losses = []
        model.train()
        for epoch in range(epochs):
            epoch_loss = 0
            for x_batch, y_batch in loader:
                optimizer.zero_grad()
                outputs = model(x_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            avg_loss = epoch_loss / len(loader)
            losses.append(avg_loss)
            if (epoch + 1) % 5 == 0:
                print(f"    Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")
        return losses

    # 4. 执行对比
    N, C, L = 512, 1, 32 # 使用单通道以简化问题
    X, Y = create_classification_data(N, C, L)
    dataset = TensorDataset(X, Y)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    print("--- 训练基线模型 (无 IterNorm)... ---")
    baseline_model = SimpleCNN(C, L, use_iternorm=False)
    baseline_losses = train_model(baseline_model, loader)

    print("\n--- 训练实验模型 (有 IterNorm)... ---")
    exp_model = SimpleCNN(C, L, use_iternorm=True)
    exp_losses = train_model(exp_model, loader)
    
    print("\n--- 实验结论 ---")
    final_baseline_loss = baseline_losses[-1]
    final_exp_loss = exp_losses[-1]
    
    print(f"基线模型最终损失: {final_baseline_loss:.6f}")
    print(f"实验模型最终损失: {final_exp_loss:.6f}")
    
    if final_exp_loss < final_baseline_loss:
        print("结论: IterNormIndividualAuto 加速了模型收敛并/或达到了更低的损失，证明了其在下游任务中的有效性。")
    else:
        print("结论: 在此任务中，IterNormIndividualAuto 未能展现明显优势。")

# run_experiment_4()


In [17]:
run_experiment_4()


========================= 实验四: 下游任务收敛性对比实验 =========================
--- 训练基线模型 (无 IterNorm)... ---
    Epoch 5/20, Loss: 1.067571
    Epoch 10/20, Loss: 1.008163
    Epoch 15/20, Loss: 0.922987
    Epoch 20/20, Loss: 0.816964

--- 训练实验模型 (有 IterNorm)... ---
    Epoch 5/20, Loss: 1.046686
    Epoch 10/20, Loss: 0.999737
    Epoch 15/20, Loss: 0.940904
    Epoch 20/20, Loss: 0.878733

--- 实验结论 ---
基线模型最终损失: 0.816964
实验模型最终损失: 0.878733
结论: 在此任务中，IterNormIndividualAuto 未能展现明显优势。


In [23]:
# verification_experiment.py
import torch
import torch.nn as nn
import sys
import os
sys.path.append('..')

# 导入原始和新修改的模块
from normailzation.IterNorm_temp import IterNormTemp # 假设原始文件名为 IterNorm_temp.py
from normailzation.IterNormTempAuto import IterNormTempAuto

def whitening_check(output_tensor, seq_length):
    """检查输出张量的白化程度"""
    z = output_tensor.permute(2, 0, 1).contiguous().view(seq_length, -1)
    covariance = torch.matmul(z, z.t()) / z.size(1)
    eye = torch.eye(seq_length, device=output_tensor.device)
    
    # 对角线元素应接近 1
    diag_mean = torch.diag(covariance).mean()
    # 非对角线元素绝对值之和应接近 0
    # off_diag_sum = torch.sum(torch.abs(covariance - torch.diag(torch.diag(covariance))))
    # 计算非对角线元素的均值
    off_diag = covariance - torch.diag(torch.diag(covariance))
    off_diag_mean = off_diag.abs().sum() / (seq_length * (seq_length - 1))
    
    return diag_mean, off_diag_mean, covariance


# --- 1. 实验设置 ---
print("--- 1. Experimental Setup ---")
batch_size = 16
num_channels = 8
seq_length = 32
T = 7
eps = 1e-5
momentum = 0.9 # 使用较高的动量以在单次迭代中显著更新 running_stats
affine = True # 测试仿射变换层

# 确保两个模型在完全相同的条件下初始化
torch.manual_seed(42)
original_model = IterNormTemp(seq_len=seq_length, T=T, eps=eps, momentum=momentum, affine=affine)

torch.manual_seed(42) # 重置种子以保证初始化一致
autograd_model = IterNormTempAuto(seq_len=seq_length, T=T, eps=eps, momentum=momentum, affine=affine)

# 设置为训练模式
original_model.train()
autograd_model.train()

# 创建相同的随机输入
input_tensor = torch.randn(batch_size, num_channels, seq_length)
input_original = input_tensor.clone().requires_grad_()
input_autograd = input_tensor.clone().requires_grad_()

print(f"Input tensor shape: {input_tensor.shape}")
print("-" * 30)

# --- 2. 前向传播等价性验证 ---
print("\n--- 2. Forward Pass Equivalence Check ---")
output_original = original_model(input_original)
output_autograd = autograd_model(input_autograd)

# 比较两个输出的接近程度
forward_pass_close = torch.allclose(output_original, output_autograd, atol=1e-6)
print(f"Forward outputs are close: {forward_pass_close}")
print(f"Max absolute difference: {(output_original - output_autograd).abs().max().item()}")
assert forward_pass_close, "Forward pass outputs do not match!"
print("-" * 30)


# --- 3. 反向传播等价性验证 ---
print("\n--- 3. Backward Pass (Gradient) Equivalence Check ---")

# 使用相同的标量损失函数
loss_original = output_original.mean()
loss_autograd = output_autograd.mean()

# 分别进行反向传播
loss_original.backward()
loss_autograd.backward()

# 比较输入的梯度
grad_close = torch.allclose(input_original.grad, input_autograd.grad, atol=1e-6)
print(f"Input gradients are close: {grad_close}")
print(f"Max absolute difference in gradients: {(input_original.grad - input_autograd.grad).abs().max().item()}")

# 比较仿射参数的梯度
if affine:
    weight_grad_close = torch.allclose(original_model.weight.grad, autograd_model.weight.grad, atol=1e-6)
    bias_grad_close = torch.allclose(original_model.bias.grad, autograd_model.bias.grad, atol=1e-6)
    print(f"Weight gradients are close: {weight_grad_close}")
    print(f"Bias gradients are close: {bias_grad_close}")
    assert grad_close and weight_grad_close and bias_grad_close, "Backward pass gradients do not match!"
else:
    assert grad_close, "Backward pass gradients do not match!"

print("-" * 30)

# --- 4. 白化有效性验证 ---
print("\n--- 4. Whitening Property Check (for Autograd version) ---")
diag_mean_input, off_diag_mean_input, cov_matrix_input = whitening_check(input_original, seq_length)
print(f"Input covariance diagonal mean (should be ~1.0): {diag_mean_input.item():.6f}")
print(f"Input covariance off-diagonal absolute mean (should be ~0.0): {off_diag_mean_input.item():.6f}")
diag_mean, off_diag_mean, cov_matrix = whitening_check(output_autograd, seq_length)
print(f"Output covariance diagonal mean (should be ~1.0): {diag_mean.item():.6f}")
print(f"Output covariance off-diagonal absolute mean (should be ~0.0): {off_diag_mean.item():.6f}")
# 验证协方差矩阵是否接近单位矩阵
# eye = torch.eye(seq_length, device=input_tensor.device)
# assert torch.allclose(cov_matrix, eye, atol=1e-2), "Whitening failed!"
# print("Whitening property verified.")
print("-" * 30)

# --- 5. 推理模式验证 ---
print("\n--- 5. Inference Mode (eval()) Check ---")
original_model.eval()
autograd_model.eval()

# 创建新的测试数据
eval_tensor = torch.randn(batch_size, num_channels, seq_length)

# 使用更新后的 running_stats 进行推理
eval_out_original = original_model(eval_tensor)
eval_out_autograd = autograd_model(eval_tensor)

eval_pass_close = torch.allclose(eval_out_original, eval_out_autograd, atol=1e-6)
print(f"Inference outputs are close: {eval_pass_close}")
print(f"Max absolute difference in inference: {(eval_out_original - eval_out_autograd).abs().max().item()}")
assert eval_pass_close, "Inference mode outputs do not match!"
print("Inference mode verified.")
print("-" * 30)

--- 1. Experimental Setup ---
Input tensor shape: torch.Size([16, 8, 32])
------------------------------

--- 2. Forward Pass Equivalence Check ---
Forward outputs are close: True
Max absolute difference: 0.0
------------------------------

--- 3. Backward Pass (Gradient) Equivalence Check ---
Input gradients are close: True
Max absolute difference in gradients: 1.266850374603834e-10
Weight gradients are close: True
Bias gradients are close: True
------------------------------

--- 4. Whitening Property Check (for Autograd version) ---
Input covariance diagonal mean (should be ~1.0): 0.992028
Input covariance off-diagonal absolute mean (should be ~0.0): 0.069473
Output covariance diagonal mean (should be ~1.0): 0.988191
Output covariance off-diagonal absolute mean (should be ~0.0): 0.003376
------------------------------

--- 5. Inference Mode (eval()) Check ---
Inference outputs are close: True
Max absolute difference in inference: 0.0
Inference mode verified.
------------------------

In [26]:
def create_correlated_data(batch_size, num_channels, seq_len, device):
    """
    生成在时间维度(seq_len)上具有强相关性的数据。
    """
    print("--- 0.a. Generating Correlated Input Data ---")
    
    # 1. 创建一个随机矩阵A，并由此构造一个非单位阵的目标协方差矩阵
    torch.manual_seed(123) # 固定种子以保证协方差矩阵可复现
    A = torch.randn(seq_len, seq_len, device=device)
    # 构造一个对称半正定矩阵
    target_cov = A @ A.T
    # 加上一个小的单位矩阵以保证其正定，从而保证 Cholesky 分解的数值稳定性
    target_cov += torch.eye(seq_len, device=device) * 1e-3
    # 归一化，使其迹为 seq_len，类似于单位阵
    target_cov = target_cov / target_cov.trace() * seq_len
    
    # 2. 对目标协方差矩阵进行Cholesky分解
    L = torch.linalg.cholesky(target_cov)
    
    # 3. 生成独立同分布的原始数据
    z = torch.randn(seq_len, batch_size * num_channels, device=device)
    
    # 4. 通过L变换，为数据引入相关性
    # correlated_flat 的形状为 (L, B*C)
    correlated_flat = L @ z
    
    # 5. 恢复为 (B, C, L) 的形状
    correlated_tensor = correlated_flat.view(seq_len, batch_size, num_channels).permute(1, 2, 0).contiguous()
    
    print("Correlated data generated.")
    return correlated_tensor, target_cov


def whitening_check(data_tensor, seq_length):
    """计算并返回数据在时间维度上的协方差矩阵及其统计量"""
    # 将 (B, C, L) -> (L, B*C)
    z = data_tensor.detach().permute(2, 0, 1).contiguous().view(seq_length, -1)
    covariance = torch.matmul(z, z.t()) / z.size(1)
    diag_mean = torch.diag(covariance).mean()
    off_diag_sum = torch.sum(torch.abs(covariance - torch.diag(torch.diag(covariance))))
    return diag_mean, off_diag_sum, covariance

# --- 实验设置 ---
batch_size = 32
num_channels = 8
seq_length = 16
T = 9
eps = 1e-5
momentum = 0.9 
affine = False # 为了更纯粹地观察白化效果，暂时关闭仿射变换
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
autograd_model = IterNormTempAuto(seq_len=seq_length, T=T, eps=eps, momentum=momentum, affine=affine).to(device)
autograd_model.train()

# --- 0. 生成相关性数据并进行检查 ---
input_tensor, target_cov = create_correlated_data(batch_size, num_channels, seq_length, device)
input_tensor.requires_grad_()

print("\n--- 0.b. Input Data Correlation Check (BEFORE Whitening) ---")
input_diag_mean, input_off_diag_sum, input_cov = whitening_check(input_tensor, seq_length)
print(f"Input Covariance Diagonal Mean: {input_diag_mean.item():.4f}")
print(f"Input Covariance Off-Diagonal Absolute Sum (should be LARGE): {input_off_diag_sum.item():.4f}")
# 验证输入数据的协方差确实不是单位矩阵
is_not_identity = not torch.allclose(input_cov, torch.eye(seq_length, device=device), atol=0.1)
print(f"Is input covariance NOT an identity matrix? {is_not_identity}")
assert is_not_identity, "Generated data is not correlated enough!"
print("-" * 40)


# --- 1. 前向传播与白化效果验证 ---
print("\n--- 1. Whitening Effect Verification (AFTER Whitening) ---")
output_autograd = autograd_model(input_tensor)

output_diag_mean, output_off_diag_sum, output_cov = whitening_check(output_autograd, seq_length)
print(f"Output Covariance Diagonal Mean (should be ~1.0): {output_diag_mean.item():.4f}")
print(f"Output Covariance Off-Diagonal Absolute Sum (should be ~0.0): {output_off_diag_sum.item():.4f}")

# 验证输出协方差矩阵是否接近单位矩阵
is_identity = torch.allclose(output_cov, torch.eye(seq_length, device=device), atol=1e-2)
print(f"Is output covariance an identity matrix? {is_identity}")
assert is_identity, "Whitening effect failed!"
print("Whitening effect strongly verified.")
print("-" * 40)

# --- 2. 梯度检查 ---
print("\n--- 2. Gradient Sanity Check ---")
# 对输出求和作为损失
loss = output_autograd.sum()
loss.backward()
grad_norm = input_tensor.grad.norm()
print(f"Gradient norm w.r.t input: {grad_norm.item()}")
assert grad_norm > 0, "Gradient is zero, backward pass might have issues."
print("Backward pass seems to be working.")
print("-" * 40)

--- 0.a. Generating Correlated Input Data ---
Correlated data generated.

--- 0.b. Input Data Correlation Check (BEFORE Whitening) ---
Input Covariance Diagonal Mean: 0.9967
Input Covariance Off-Diagonal Absolute Sum (should be LARGE): 48.2122
Is input covariance NOT an identity matrix? True
----------------------------------------

--- 1. Whitening Effect Verification (AFTER Whitening) ---
Output Covariance Diagonal Mean (should be ~1.0): 0.9332
Output Covariance Off-Diagonal Absolute Sum (should be ~0.0): 7.0127
Is output covariance an identity matrix? False


AssertionError: Whitening effect failed!